# QUROCR: Open-Source OCR Benchmark Evaluation (Colab GPU/CPU)
This Google Colab notebook evaluates traditional and neural open-source OCR engines on the **QUROCR Benchmark-200** dataset (200 Nastaliq text-line crops).

### Models Evaluated:
1. **Tesseract OCR v5** (with Urdu language data: `urd`)
2. **PaddleOCR v2.7+** (Multilingual recognition for Arabic/Urdu scripts)
3. **EasyOCR** (PyTorch-based Deep Learning Urdu OCR)

This runs completely **FREE** on Google Colab (no API keys required).

In [ ]:
# Cell 1: Install System Dependencies & Urdu Language Packs
!sudo apt-get update -qq
!sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-urd tesseract-ocr-ara libtesseract-dev
!pip install -q pytesseract paddlepaddle paddleocr easyocr tqdm

import os, zipfile, csv, re, json, time
import pytesseract
from PIL import Image

# Verify Tesseract Urdu installation
langs = pytesseract.get_languages()
print('Tesseract Available Languages:', langs)
assert 'urd' in langs, 'Urdu language data not found for Tesseract!'

In [ ]:
# Cell 2: Unpack Benchmark Dataset
if os.path.exists('benchmark200.zip'):
    with zipfile.ZipFile('benchmark200.zip', 'r') as z:
        z.extractall('benchmark200')
    print('Extracted benchmark200.zip successfully!')
else:
    print('Please upload benchmark200.zip using the Colab file browser.')

os.makedirs('results_opensource', exist_ok=True)


In [ ]:
# Cell 3: Metrics & Sanitization Utilities
def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]

def clean_urdu(text):
    if not text: return ''
    text = re.sub(r'[a-zA-Z]', '', text) # Strict NO English filter
    text = re.sub(r'[\#\*\_\~\`\>\[\]\(\)\{\}]', '', text)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return ' '.join(lines).strip()

def load_benchmark_labels():
    labels_p = 'benchmark200/labels200.csv'
    if not os.path.exists(labels_p): labels_p = 'labels200.csv'
    with open(labels_p, 'r', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))


In [ ]:
# Cell 4: Evaluate Tesseract OCR
from tqdm.notebook import tqdm

items = load_benchmark_labels()
img_dir = 'benchmark200/images' if os.path.exists('benchmark200/images') else 'images'
tess_results = []

print(f'Running Tesseract OCR (urd) on {len(items)} samples...')
custom_config = r'--oem 1 --psm 7 -l urd'

for item in tqdm(items):
    fname = item['filename']
    ref = item['reference']
    src = item['source']
    img_p = os.path.join(img_dir, fname)
    try:
        raw = pytesseract.image_to_string(Image.open(img_p), config=custom_config)
        clean = clean_urdu(raw)
    except Exception as e:
        clean = ''
    cer = levenshtein(clean, ref) / max(len(ref), 1) * 100
    p_words, r_words = clean.split(), ref.split()
    wer = levenshtein(p_words, r_words) / max(len(r_words), 1) * 100
    tess_results.append({
        'filename': fname, 'prediction': clean, 'reference': ref,
        'source': src, 'cer': round(cer, 2), 'wer': round(wer, 2)
    })

with open('results_opensource/predictions_Tesseract_urd.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'prediction', 'reference', 'source', 'cer', 'wer'])
    writer.writeheader()
    writer.writerows(tess_results)
print('Tesseract evaluation saved to results_opensource/predictions_Tesseract_urd.csv')


In [ ]:
# Cell 5: Evaluate PaddleOCR (Arabic / Multilingual)
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(use_angle_cls=True, lang='arabic', show_log=False)
paddle_results = []

print(f'Running PaddleOCR on {len(items)} samples...')
for item in tqdm(items):
    fname = item['filename']
    ref = item['reference']
    src = item['source']
    img_p = os.path.join(img_dir, fname)
    try:
        result = ocr_engine.ocr(img_p, cls=True)
        texts = [line[1][0] for res in result if res for line in res]
        raw = ' '.join(texts)
        clean = clean_urdu(raw)
    except Exception as e:
        clean = ''
    cer = levenshtein(clean, ref) / max(len(ref), 1) * 100
    p_words, r_words = clean.split(), ref.split()
    wer = levenshtein(p_words, r_words) / max(len(r_words), 1) * 100
    paddle_results.append({
        'filename': fname, 'prediction': clean, 'reference': ref,
        'source': src, 'cer': round(cer, 2), 'wer': round(wer, 2)
    })

with open('results_opensource/predictions_PaddleOCR.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'prediction', 'reference', 'source', 'cer', 'wer'])
    writer.writeheader()
    writer.writerows(paddle_results)
print('PaddleOCR evaluation saved to results_opensource/predictions_PaddleOCR.csv')


In [ ]:
# Cell 6: Evaluate EasyOCR (Urdu)
import easyocr

reader = easyocr.Reader(['ur'], gpu=True) # Set gpu=False if running on CPU
easy_results = []

print(f'Running EasyOCR (ur) on {len(items)} samples...')
for item in tqdm(items):
    fname = item['filename']
    ref = item['reference']
    src = item['source']
    img_p = os.path.join(img_dir, fname)
    try:
        res = reader.readtext(img_p, detail=0)
        raw = ' '.join(res)
        clean = clean_urdu(raw)
    except Exception as e:
        clean = ''
    cer = levenshtein(clean, ref) / max(len(ref), 1) * 100
    p_words, r_words = clean.split(), ref.split()
    wer = levenshtein(p_words, r_words) / max(len(r_words), 1) * 100
    easy_results.append({
        'filename': fname, 'prediction': clean, 'reference': ref,
        'source': src, 'cer': round(cer, 2), 'wer': round(wer, 2)
    })

with open('results_opensource/predictions_EasyOCR_ur.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'prediction', 'reference', 'source', 'cer', 'wer'])
    writer.writeheader()
    writer.writerows(easy_results)
print('EasyOCR evaluation saved to results_opensource/predictions_EasyOCR_ur.csv')


In [ ]:
# Cell 7: Comparative Metrics Summary
import glob

def print_summary(csv_p):
    with open(csv_p, 'r', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    def compute(sub, tag):
        c_dist = sum(levenshtein(r['prediction'], r['reference']) for r in sub)
        c_len = sum(len(r['reference']) for r in sub)
        cer = (c_dist / c_len * 100) if c_len else 0
        w_dist = sum(levenshtein(r['prediction'].split(), r['reference'].split()) for r in sub)
        w_len = sum(len(r['reference'].split()) for r in sub)
        wer = (w_dist / w_len * 100) if w_len else 0
        ser = sum(1 for r in sub if r['prediction'].strip() != r['reference'].strip()) / len(sub) * 100
        acc = sum(1 for r in sub if r['prediction'].strip() == r['reference'].strip()) / len(sub) * 100
        return f"{tag:16}: CER={cer:5.2f}% | WER={wer:5.2f}% | SER={ser:5.2f}% | Acc={acc:5.2f}%"
    print(f"\n{'='*70}\nFile: {os.path.basename(csv_p)}\n{'-'*70}")
    print(compute(rows, 'Overall (200)'))
    print(compute([r for r in rows if r['source'] == 'real'], 'Real (60)'))
    print(compute([r for r in rows if r['source'] == 'synthetic'], 'Synthetic (140)'))

for f in sorted(glob.glob('results_opensource/*.csv')):
    print_summary(f)


In [ ]:
# Cell 8: Package & Download All Results
!zip -r results_opensource.zip results_opensource/
from google.colab import files
files.download('results_opensource.zip')
